In [1]:
# Imports
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import gc

### preprocess variants

In [ ]:
def load_and_process(parquet_file, mapping_file, prefix):
    # load
    variants = pd.read_parquet(parquet_file)
    
    # mapping
    indiv_id_mapping = pd.read_table(mapping_file).set_index('sample_id')
    
    # add indiv_id
    variants["indiv_id"] = (
        variants["sample_id"]
        .map(indiv_id_mapping["indiv_id"])
        .fillna("")
        .astype(str)
    )
    
    # ADD PREFIX HERE
    variants["indiv_id"] = prefix + variants["indiv_id"]
    
    # total counts
    variants["total_counts"] = variants["ref_counts"] + variants["alt_counts"]
    
    # filtering
    passed_read_count = (
        (variants["total_counts"] > 20) &
        (variants["ref_counts"] > 0) &
        (variants["alt_counts"] > 0)
    )
    
    return variants.loc[passed_read_count].copy()

In [ ]:
dnase_file = '/net/seq/data2/projects/nasi4/ENCODE4/dnase-cavs.v5/output/non_aggregated.all.parquet' 
atac_file = '/net/seq/data2/projects/nasi4/ENCODE4/atac-cavs.v6/output/non_aggregated.all.parquet' 
dnase_map_file = '/net/seq/data2/projects/nasi4/ENCODE4/dnase-cavs.v5/meta+sample_ids.tsv' 
atac_map_file = '/net/seq/data2/projects/nasi4/ENCODE4/atac-cavs.v6/meta+sample_ids.tsv'

In [ ]:
dnase_variants = load_and_process(dnase_file, dnase_map_file, "DNASE_")
atac_variants  = load_and_process(atac_file, atac_map_file, "ATAC_")

In [ ]:
combined_variants = pd.concat(
    [dnase_variants, atac_variants],
    axis=0,
    ignore_index=True
)

#### Remove Duplicates and Multiple alleles

In [ ]:
#remove duplicate rows
key_cols = ["#chr", "start", "end","ref", "alt", "indiv_id","sample_id"]

variants_no_duplicate = combined_variants.drop_duplicates(subset=key_cols, keep=False)

In [ ]:
#remove places where individual ID has more than one variant at one position with different ref/alt allele
key_cols = ["#chr", "start", "end", "indiv_id"]

# Count unique allele combinations per individual per position
allele_counts = (
    variants_no_duplicate
    .groupby(key_cols)[["ref", "alt"]]
    .nunique()
)

# positions where ref or alt varies (multi-allelic)
multi_mask = (allele_counts["ref"] > 1) | (allele_counts["alt"] > 1)

multi_sites = allele_counts[multi_mask].index

# Remove those rows
variants_no_multiple = variants_no_duplicate[
    ~variants_no_duplicate.set_index(key_cols).index.isin(multi_sites)
].reset_index(drop=True)

In [ ]:
variants_no_multiple.to_parquet('/home/mbrannon/tmp/5_7_variantsnomulti_atac_plus_dnase.parquet')

### Combine genotype files

In [ ]:
#sbatch /home/mbrannon/.local/src/vinson/scripts/cleangenotype_phased.sh
#outputs unzipped and indiv with prefix for atac and dnase genotype file, removes multiallelic

In [ ]:
# cat /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/atac.phased.clean.tsv /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/dnase.phased.clean.tsv > /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/genotype.phased.combined.tsv

# sort -k1,1 -k2,2n /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/genotype.phased.combined.tsv > /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/genotype.combined.sorted.bed

# bgzip -f /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/genotype.combined.sorted.bed

# tabix -p bed /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/genotype.combined.sorted.bed.gz

### make sure matches genotype file (merge)

In [ ]:
used = pd.read_parquet(
    "/home/mbrannon/tmp/5_7_variantsnomulti_atac_plus_dnase.parquet"
)
# used = variants_no_multiple

key_cols = ["#chr", "start", "end", "ref", "alt", "indiv_id"]

print("[INFO] Loading cleaned genotype files")

geno_cols = [
    "#chr",
    "start",
    "end",
    "ref",
    "alt",
    "indiv_id",
    "gt"
]

dtype_map = {
    "#chr": str,
    "start": str,
    "end": str,
    "ref": str,
    "alt": str,
    "indiv_id": str,
    "gt": str
}
#this should be output from concat and sort above
genotype_file = '/net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/genotype.phased.combined.tsv > /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/genotype.combined.sorted.bed'
geno = pd.read_csv(genotype_file,
    sep="\t",
    names=geno_cols,   
    dtype=dtype_map,
)

# -------------------------
# Normalize dtypes + chr naming
# -------------------------

for df in [used, geno]:

    df["#chr"] = (
        df["#chr"]
        .astype(str)
        # .str.replace("^chr", "", regex=True)
    )

    df["start"] = pd.to_numeric(
        df["start"]
        .astype(str)
    )

    df["end"] = pd.to_numeric(
        df["end"]
        .astype(str)
    )

    df["ref"] = df["ref"].astype(str)

    df["alt"] = df["alt"].astype(str)

    df["indiv_id"] = df["indiv_id"].astype(str)

print("[INFO] used dtypes")
print(used[key_cols].dtypes)

print("[INFO] geno dtypes")
print(geno[key_cols].dtypes)

print("[INFO] Merging")

merged = used.merge(
    geno,
    on=key_cols,
    how="inner",
)

print("[INFO] Final merged shape:", merged.shape)
#dont need to save intermediate parquet can use directly in next part
merged.to_parquet(
    '/home/mbrannon/tmp/5_7_both_atac_dnase_variants_phased_passedgeno.parquet'
)



### make adata

In [ ]:
#this code also exists as python script here: /home/mbrannon/.local/src/vinson/scripts/script_makeadata.py
#usually srun with 300G

In [ ]:
#paths
#output from previous step dataframe named merged
PARQUET_PATH = "/home/mbrannon/tmp/5_7_both_atac_dnase_variants_phased_passedgeno.parquet"

DHS_PATH = "/net/seq/data2/projects/ENCODE4Plus/REGULOME/one_big_beautiful_index/latest.reference_anndata.zarr"
ATAC_PATH = "/net/seq/data2/projects/ENCODE4Plus/REGULUME/one_big_beautiful_index/latest.SRA_anndata.zarr"

OUT_DIR = "/home/mbrannon/tmp/pipeline_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
#used_results_raw = pd.read_parquet(PARQUET_PATH)
used_results = merged
dhsadata = read_zarr_backed(DHS_PATH)
atac_anndata = read_zarr_backed(ATAC_PATH)

# =========================================================
# Column setup
# =========================================================

layer_cols_numeric = [
    "ref_counts",
    "alt_counts",
    "BAD",
    "logit_es",
    "FDR_sample",
    "inverse_mse",
    "total_counts",
]

used_results_raw[layer_cols_numeric] = used_results_raw[layer_cols_numeric].astype(float)

# =========================================================
# Variant ID
# =========================================================

used_results_raw["variant_id"] = (
    used_results_raw["#chr"]
    + ":"
    + used_results_raw["end"].astype(str)
    + ":"
    + used_results_raw["ref"]
    + ":"
    + used_results_raw["alt"]
)

used_results_raw["calc_alt_counts"] = (
    used_results_raw["total_counts"] - used_results_raw["ref_counts"]
)

# =========================================================
# Base masks (global, reused)
# =========================================================

passed = (
    (used_results_raw["total_counts"] > 20)
    & (used_results_raw["ref_counts"] > 0)
    & (used_results_raw["calc_alt_counts"] > 0)
)

pos_mask = passed & (used_results_raw["FDR_sample"] < 0.1) & (used_results_raw["logit_es"].abs() < 6)
neg_mask = passed & (used_results_raw["FDR_sample"] > 0.5)

pos_df = used_results_raw[pos_mask]
neg_df = used_results_raw[neg_mask]

print("Pos:", len(pos_df), "Neg:", len(neg_df))

# =========================================================
# Split datasets (DO NOT MODIFY RAW)
# =========================================================

atac_df = used_results_raw.query(
    'indiv_id.str.startswith("ATAC")',
    engine="python"
)

dnase_df = used_results_raw.query(
    'indiv_id.str.startswith("DNASE")',
    engine="python"
)

both_df = used_results_raw

# atac_bad15 = atac_df.query("BAD <= 1.5")
# dnase_bad15 = dnase_df.query("BAD <= 1.5")
both_bad15 = both_df.query("BAD <= 1.5")

# =========================================================
# OBS (shared across ALL datasets)
# =========================================================

ref_obs = dhsadata.obs.copy()
atac_obs = atac_anndata.obs.copy()

ref_obs.index = ref_obs.index.astype(str)
atac_obs.index = atac_obs.index.astype(str)

obs = pd.concat([ref_obs, atac_obs])

obs["modality"] = np.where(
    obs.index.str.startswith("AG"),
    "DNASE",
    "ATAC",
)
# =========================================================
# VAR (shared)
# =========================================================

var_df = (
    pd.concat([pos_df, neg_df])
    .drop_duplicates("variant_id")
    .set_index("variant_id")[
        ["#chr", "start", "end", "ID", "ref", "alt", "AAF", "RAF"]
    ]
)

# =========================================================
# GLOBAL validation split (500 samples total)
# =========================================================

n_val = 500

n_atac_val = int(round((obs["modality"] == "ATAC").mean() * n_val))
n_dnase_val = n_val - n_atac_val

val_atac = (
    obs.query('modality == "ATAC"')
    .sample(n=n_atac_val, random_state=42)
    .index
)

val_dnase = (
    obs.query('modality == "DNASE"')
    .sample(n=n_dnase_val, random_state=42)
    .index
)

val_samples = np.concatenate([val_atac, val_dnase])

# =========================================================
# Indexers
# =========================================================

obs_indexer = pd.Series(range(len(obs)), index=obs.index)
var_indexer = pd.Series(range(len(var_df)), index=var_df.index)

# =========================================================
# Embeddings (shared)
# =========================================================

ref_emb = pd.DataFrame(
    dhsadata.obsm["motif_embeddings"],
    index=dhsadata.obs_names
)

atac_emb = pd.DataFrame(
    atac_anndata.obsm["motif_embeddings"],
    index=atac_anndata.obs_names
)

ref_emb.index = ref_emb.index.astype(str)
atac_emb.index = atac_emb.index.astype(str)

combined_emb = pd.concat([ref_emb, atac_emb]).reindex(obs.index)

del ref_emb
del atac_emb
del dhsadata
del atac_anndata
gc.collect()

In [ ]:
# =========================================================
# Builder
# =========================================================

def build_dataset(df, name, out_path):

    print(f"\n================ {name} ================")

    # =====================================================
    # Dataset-specific positives / negatives
    # =====================================================

    df = df.copy()

    df["calc_alt_counts"] = (
        df["total_counts"] - df["ref_counts"]
    )

    passed_local = (
        (df["total_counts"] > 20)
        & (df["ref_counts"] > 0)
        & (df["calc_alt_counts"] > 0)
    )

    pos_df_local = df[
        passed_local
        & (df["FDR_sample"] < 0.1)
        & (df["logit_es"].abs() < 6)
    ]

    neg_df_local = df[
        passed_local
        & (df["FDR_sample"] > 0.5)
    ]

    print("Positives:", len(pos_df_local))
    print("Negatives:", len(neg_df_local))

    if len(pos_df_local) == 0:
        raise ValueError(f"{name}: no positive examples")

    if len(neg_df_local) < len(pos_df_local):
        raise ValueError(
            f"{name}: negatives ({len(neg_df_local)}) "
            f"< positives ({len(pos_df_local)})"
        )

    # =====================================================
    # Create AnnData
    # =====================================================

    adata = ad.AnnData(
        X=None,
        obs=obs.copy(),
        var=var_df.copy()
    )

    obs_idx = pd.Series(
        range(len(adata.obs)),
        index=adata.obs_names
    )

    var_idx = pd.Series(
        range(len(adata.var)),
        index=adata.var_names
    )

    adata.obsm["motif_embeddings"] = combined_emb

    epochs = ["epoch_1", "epoch_2", "epoch_3"]

    # =====================================================
    # Build sparse layers
    # =====================================================

    for i, epoch in enumerate(epochs):

        print(f"\n========== {epoch} ==========")

        neg_sampled = neg_df_local.sample(
            n=len(pos_df_local),
            random_state=100 + i
        )

        epoch_df = pd.concat(
            [pos_df_local, neg_sampled],
            axis=0
        ).sample(
            frac=1,
            random_state=100 + i
        )

        row_idx = epoch_df["sample_id"].map(obs_idx).to_numpy()
        col_idx = epoch_df["variant_id"].map(var_idx).to_numpy()

        missing_rows = np.isnan(row_idx).sum()
        missing_cols = np.isnan(col_idx).sum()

        print("Missing rows:", missing_rows)
        print("Missing cols:", missing_cols)

        keep = (
            ~np.isnan(row_idx)
            & ~np.isnan(col_idx)
        )

        row_idx = row_idx[keep].astype(np.int32)
        col_idx = col_idx[keep].astype(np.int32)

        epoch_df = epoch_df.iloc[keep]

        for col in layer_cols_numeric:

            print(f"[BUILD] {col}.{epoch}")

            vals = epoch_df[col].to_numpy()

            mat = sp.coo_matrix(
                (vals, (row_idx, col_idx)),
                shape=adata.shape
            ).tocsr()

            adata.layers[f"{col}.{epoch}"] = mat

        del epoch_df
        del neg_sampled
        gc.collect()

    # =====================================================
    # metadata
    # =====================================================

    mapping = (
        used_results_raw[
            ["sample_id", "indiv_id"]
        ]
        .dropna()
        .drop_duplicates("sample_id")
        .set_index("sample_id")["indiv_id"]
    )

    adata.obsm["indiv_id"] = (
        adata.obs_names
        .map(mapping)
        .fillna("")
        .to_numpy()
        .reshape(-1, 1)
    )

    # =====================================================
    # sample split
    # =====================================================

    if name.startswith("atac"):
        dataset_val_samples = val_atac

    elif name.startswith("dnase"):
        dataset_val_samples = val_dnase

    else:
        dataset_val_samples = val_samples

    adata.obsm["split_data"] = np.where(
        adata.obs_names.isin(dataset_val_samples),
        "val",
        "train"
    )
    

    # =====================================================
    # chromosome split
    # =====================================================

    validation_chroms = ["chr9", "chr22"]

    adata.varm["split_data"] = np.where(
        adata.var["#chr"].isin(validation_chroms),
        "val",
        "train"
    )

    # =====================================================
    # metadata
    # =====================================================

    

    adata.uns["val_samples"] = val_samples.tolist()
    adata.uns["epoch_names"] = epochs

    # =====================================================
    # save
    # =====================================================

    print("Saving:", out_path)

    adata.write_h5ad(
        out_path,
        compression="gzip"
    )

    del adata
    gc.collect()

In [ ]:
# =========================================================
# RUN ALL 6 DATASETS
# =========================================================

datasets = [
    # ("atac", atac_df),
    # ("dnase", dnase_df),
    ("both", both_df),

    # ("atac_bad15", atac_bad15),
    # ("dnase_bad15", dnase_bad15),
    ("both_bad15", both_bad15),
]

In [ ]:
for name, df in datasets:
    build_dataset(df, name, OUT_DIR + f"6_5_nonphased_{name}.h5ad")

### command for running training

In [ ]:
python submit_to_sbatch.py \
/net/seq/data2/projects/mbrannon/6_2_both.h5ad \
/net/seq/data/genomes/human/GRCh38/noalts/GRCh38_no_alts.fa \
/net/seq/data2/projects/mbrannon/variant_model_paper \
--config /home/mbrannon/.local/src/vinson/train/variant/legnetAPR09.yaml \
--genotype_file /net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/phased_genotype_dnase_and_atac.bed.gz \
--sequence_model_checkpoint /home/mbrannon/.local/src/vinson/train/variant/legnetapr09.ckpt \
--model_type variant \
--env_path /home/mbrannon/.local/miniconda3/envs/pytorch_clone \
--mem 200G \
--epochs 15 \
--nodelist hpcg04-heavy 